# M1.1 · Linear regression: kapalı form ve gradient descent

İlk öğrenen model. Üç noktalık bir veri setinde bir gradient descent adımını elle atıyor, aynı adımı döngüyle ve NumPy'da yazıyor, pandas DataFrame'inden özellik isimleriyle model kuruyor, `nn.Linear` ile eğitiyor ve en sonda 64 farklı learning rate'i tek seferde eğitiyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m1-1-linreg.html](https://m-gonul.github.io/pytorch-lab/m1-1-linreg.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m1_1_linreg.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\", \"pandas\"}))\n\n    def step():\n        out = need(ns, \"gd_step\")([1, 2, 3], [2, 4, 5], 0.0, 0.0, 0.1)\n        ensure(isinstance(out, (tuple, list)) and len(out) == 3, \"(w, b, kayıp) üçlüsü döndürmeli\")\n        close(out, [5 / 3, 11 / 15, 15], what=\"(w, b, kayıp)\")\n    R.test(\"elle adım: (1.667, 0.733, 15)\", step)\n\n    def cf():\n        close(need(ns, \"closed_form\")([1, 2, 3], [2, 4, 5]), [1.5, 2 / 3])\n        rng = random.Random(4)\n        xs = [rng.uniform(-3, 3) for _ in range(30)]\n        ys = [0.7 * x - 1.2 + rng.gauss(0, 0.3) for x in xs]\n        n = len(xs); xm = sum(xs) / n; ym = sum(ys) / n\n        w = sum((x - xm) * (y - ym) for x, y in zip(xs, ys)) / sum((x - xm) ** 2 for x in xs)\n        close(need(ns, \"closed_form\")(xs, ys), [w, ym - w * xm], what=\"30 rastgele nokta\")\n    R.test(\"closed_form: elle örnek ve 30 rastgele nokta\", cf)\n\n    def fit_conv():\n        close(need(ns, \"fit\")([1, 2, 3], [2, 4, 5]), [1.5, 2 / 3], rtol=0, atol=1e-4, what=\"fit sonucu\")\n        return \"2000 adımda kapalı form ile 4 basamak eşleşti\"\n    R.test(\"fit kapalı forma yakınsıyor\", fit_conv)\n\n    def loss_down():\n        w = b = 0.0\n        prev = float(\"inf\")\n        for _ in range(50):\n            w, b, L = need(ns, \"gd_step\")([1, 2, 3], [2, 4, 5], w, b, 0.05)\n            ensure(L <= prev + 1e-12, f\"kayıp arttı: {prev:.4f} → {L:.4f}\")\n            prev = L\n    R.test(\"küçük lr ile kayıp hiç artmıyor\", loss_down)\n",
 "c": "import numpy as np\nimport pandas as pd\nfrom labcheck import close, max_loops, no_imports, need, ensure, uses_any\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"normal_eq içinde döngü yok\", lambda: max_loops(src, 0, func=\"normal_eq\"))\n    R.test(\"fit_gd içinde yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"fit_gd\", why=\"örnekler üzerinde döngü olmamalı\"))\n    R.test(\"torch kullanılmıyor, inv yok\", lambda: (no_imports(src, {\"torch\"}), ensure(not uses_any(src, [\"inv\"]), \"np.linalg.inv yerine lstsq/solve kullan\")))\n    rng = np.random.default_rng(7)\n    X = rng.normal(size=(80, 4))\n    y = X @ np.array([1.5, -2.0, 0.0, 0.7]) - 1.0 + 0.05 * rng.normal(size=80)\n    Xt = np.column_stack([X, np.ones(80)])\n    th = np.linalg.solve(Xt.T @ Xt, Xt.T @ y)\n\n    def ne():\n        w, b = need(ns, \"normal_eq\")(X, y)\n        close(w, th[:4], what=\"w\"); close(b, th[4], what=\"b\")\n        w1, b1 = need(ns, \"normal_eq\")(np.array([[1.], [2.], [3.]]), np.array([2., 4., 5.]))\n        close([w1[0], b1], [1.5, 2 / 3], what=\"elle örnek\")\n    R.test(\"normal_eq: 4 özellik ve elle örnek\", ne)\n\n    def gd():\n        w, b, losses = need(ns, \"fit_gd\")(X, y, lr=0.1, steps=500)\n        losses = np.asarray(losses)\n        ensure(losses.shape == (500,), f\"losses shape {losses.shape}, (500,) bekleniyordu\")\n        ensure(np.all(np.diff(losses) <= 1e-12), \"kayıp bir adımda arttı (lr = 0.1 ile olmamalı)\")\n        close(w, th[:4], atol=1e-4, rtol=0, what=\"GD w\"); close(b, th[4], atol=1e-4, rtol=0, what=\"GD b\")\n        return f\"kayıp {losses[0]:.3f} → {losses[-1]:.5f}\"\n    R.test(\"fit_gd normal denkleme yakınsıyor, kayıp monoton\", gd)\n\n    def df_():\n        df = pd.DataFrame(X, columns=[\"a\", \"b\", \"c\", \"d\"]).assign(hedef=y)\n        w, b = need(ns, \"fit_df\")(df, \"hedef\")\n        ensure(isinstance(w, pd.Series), \"w bir pd.Series olmalı\")\n        ensure(list(w.index) == [\"a\", \"b\", \"c\", \"d\"], f\"indeks {list(w.index)}, özellik isimleri bekleniyordu\")\n        close(w.to_numpy(), th[:4]); close(b, th[4])\n        df2 = df[[\"hedef\", \"d\", \"a\", \"c\", \"b\"]]\n        w2, _ = need(ns, \"fit_df\")(df2, \"hedef\")\n        close(w2[[\"a\", \"b\", \"c\", \"d\"]].to_numpy(), th[:4], what=\"sütun sırası değişince\")\n        return \"sütun sırası değişse de isimler doğru katsayıyı taşıyor\"\n    R.test(\"fit_df: isimli Series, sütun sırasından bağımsız\", df_)\n",
 "d": "import torch\nfrom torch import nn\nfrom labcheck import close, need, ensure, uses_any\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n\n    def g():\n        out = need(ns, \"one_step_grads\")()\n        ensure(isinstance(out, (tuple, list)) and len(out) == 2, \"(gw, gb) döndürmeli\")\n        close(out, [-50 / 3, -22 / 3], rtol=1e-5)\n    R.test(\"one_step_grads: (−16.67, −7.33) — elle bulduğunla aynı\", g)\n    R.test(\"autograd + optim kullanılıyor\", lambda: ensure(uses_any(src, [\"backward\"]) and uses_any(src, [\"SGD\", \"step\"]), \"loss.backward() ve opt.step() bekleniyordu\"))\n\n    def fit():\n        torch.manual_seed(0)\n        X = torch.randn(120, 3)\n        y = X @ torch.tensor([2.0, -1.0, 0.5]) + 3 + 0.05 * torch.randn(120)\n        Xt = torch.cat([X, torch.ones(120, 1)], 1).double()\n        th = torch.linalg.lstsq(Xt, y.double().unsqueeze(1)).solution.squeeze(1)\n        m = need(ns, \"fit_torch\")(X, y, lr=0.1, steps=500)\n        ensure(isinstance(m, nn.Linear), \"nn.Linear döndürmeli\")\n        close(m.weight.detach().reshape(-1), th[:3], atol=2e-3, rtol=0, what=\"weight\")\n        close(m.bias.detach().reshape(-1), th[3:], atol=2e-3, rtol=0, what=\"bias\")\n        return \"ağırlıklar normal denklem çözümüyle 3 basamak eşleşti\"\n    R.test(\"fit_torch normal denklem çözümüne ulaşıyor\", fit)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef train_one(X, y, lr, steps):\n    w = torch.zeros(X.shape[1], dtype=X.dtype, requires_grad=True)\n    b = torch.zeros((), dtype=X.dtype, requires_grad=True)\n    for _ in range(steps):\n        loss = ((X @ w + b - y) ** 2).mean()\n        loss.backward()\n        with torch.no_grad():\n            w -= lr * w.grad; b -= lr * b.grad\n            w.grad = None; b.grad = None\n    with torch.no_grad():\n        return w.clone(), b.clone(), ((X @ w + b - y) ** 2).mean()\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"sweep içinde yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"sweep\", why=\"lr'ler üzerinde döngü olmamalı\"))\n    g = torch.Generator().manual_seed(0)\n    X = torch.randn(150, 4, generator=g)\n    y = X @ torch.tensor([1.0, -2.0, 0.5, 0.0]) + 1.0 + 0.1 * torch.randn(150, generator=g)\n    lrs = torch.tensor([0.001, 0.01, 0.05, 0.1, 0.3])\n\n    def eq():\n        W, b, losses = need(ns, \"sweep\")(X, y, lrs, steps=60)\n        ensure(tuple(W.shape) == (5, 4) and tuple(b.shape) == (5,) and tuple(losses.shape) == (5,), f\"shape'ler {tuple(W.shape)}, {tuple(b.shape)}, {tuple(losses.shape)}\")\n        for i, lr in enumerate(lrs.tolist()):\n            w1, b1, l1 = train_one(X, y, lr, 60)\n            close(W[i], w1, rtol=1e-4, atol=1e-5, what=f\"lr={lr} W\")\n            close(b[i], b1, rtol=1e-4, atol=1e-5, what=f\"lr={lr} b\")\n            close(losses[i], l1, rtol=1e-4, atol=1e-5, what=f\"lr={lr} kayıp\")\n        return \"her lr tek tek eğitilmiş modelle birebir aynı\"\n    R.test(\"5 lr: tek tek eğitimle eşleşiyor\", eq)\n    if \"sweep\" not in ns:\n        return\n    lrs64 = torch.logspace(-3, -0.5, 64)\n    t_loop = bench(lambda: [train_one(X, y, float(lr), 100) for lr in lrs64], repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"sweep\"](X, y, lrs64, steps=100), repeat=3, number=1)\n    _, _, Lf = ns[\"sweep\"](X, y, lrs64, steps=100)\n    best = int(torch.nan_to_num(Lf, nan=float(\"inf\")).argmin())\n    R.table(\"64 learning rate × 100 adım\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for lr: 64 ayrı eğitim\", fmt_time(t_loop), \"1×\"],\n             [\"senin sweep (tek tensör)\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]],\n            note=f\"En iyi lr ≈ {float(lrs64[best]):.3f} (son kayıp {float(Lf[best]):.4f}).\")\n",
 "x1": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure, uses_any\n\nFUNCS = [\"phi\", \"fit_poly\", \"rmse\", \"resid\", \"quad_sig\"]\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    for f in FUNCS:\n        R.test(f\"{f} içinde döngü yok\", lambda f=f: max_loops(src, 0, func=f))\n    R.test(\"torch kullanılmıyor, inv yok\", lambda: (need(ns, \"phi\"),\n           ensure(not uses_any(src, {\"torch\", \"inv\", \"polyfit\"}), \"lstsq kullan; inv/polyfit yasak\")))\n\n    phi = need(ns, \"phi\")\n    fit_poly = need(ns, \"fit_poly\")\n    rmse = need(ns, \"rmse\")\n    resid = need(ns, \"resid\")\n    quad_sig = need(ns, \"quad_sig\")\n\n    def phi_ok():\n        close(phi(np.array([1.0, 2.0, 3.0]), 2),\n              [[1, 1, 1], [1, 2, 4], [1, 3, 9]], rtol=0, atol=1e-12, what=\"phi([1,2,3], 2)\")\n        close(phi(np.array([2.0, -1.0]), 0), [[1], [1]], rtol=0, atol=1e-12, what=\"deg=0: birler sütunu\")\n        rng = np.random.default_rng(3)\n        v = np.linspace(-3, 3, 17)\n        close(phi(v, 4), np.vander(v, 5, increasing=True), rtol=1e-12, atol=1e-12, what=\"np.vander ile\")\n        ensure(phi(np.array([1.0]), 2).shape == (1, 3), \"shape (n, deg+1) olmalı\")\n    R.test(\"phi: elle örnek, np.vander ve shape\", phi_ok)\n\n    def fit_ok():\n        v = np.linspace(-2, 2, 25)\n        close(fit_poly(v, 3 * v + 0.8 * v**2 - 1, 2), [-1, 3, 0.8], rtol=0, atol=1e-9,\n              what=\"gürültüsüz karesel: [−1, 3, 0.8]\")\n        close(fit_poly(v, 1.5 * v - 1, 1), [-1, 1.5], rtol=0, atol=1e-9, what=\"gürültüsüz doğrusal\")\n        t = fit_poly(v, 1.5 * v - 1, 4)\n        close(t, [-1, 1.5, 0, 0, 0], rtol=0, atol=1e-7, what=\"aşırı derece: gereksiz katsayılar ~0\")\n        return \"derece artsa da gereksiz katsayılar ~0: lstsq en küçük normlu çözümü seçer\"\n    R.test(\"fit_poly: katsayıları geri kazandırır\", fit_ok)\n\n    def rmse_ok():\n        ensure(isinstance(rmse(np.array([1.0, 2.0]), np.array([1.0, 2.0])), float), \"float döndürmeli\")\n        close(rmse(np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 3.0])), 0, rtol=0, atol=1e-12, what=\"mükemmel uydurma\")\n        close(rmse(np.array([1.0, 2.0]), np.array([2.0, 1.0])), 1.0, rtol=0, atol=1e-12, what=\"[1,2] vs [2,1]\")\n        rng = np.random.default_rng(5)\n        a, b = rng.normal(size=13), rng.normal(size=13)\n        close(rmse(a, b), np.sqrt(np.mean((b - a) ** 2)), rtol=1e-12, atol=1e-12, what=\"rastgele\")\n    R.test(\"rmse: tanımla birebir\", rmse_ok)\n\n    rng = np.random.default_rng(0)\n    x = np.linspace(-2, 2, 40)\n    y = 3 * x + 0.8 * x**2 - 1 + 0.4 * rng.normal(size=40)\n\n    def resid_ok():\n        v = np.array([1.0, 2.0, 3.0])\n        r = resid(v, np.array([2.0, 4.0, 5.0]), 1)\n        close(r, [1 / 6, -1 / 3, 1 / 6], rtol=0, atol=1e-9, what=\"r = ŷ − y (l4 işareti)\")\n    R.test(\"resid: r = ŷ − y işareti\", resid_ok)\n\n    def sig_ok():\n        v = np.linspace(-2, 2, 31)\n        close(quad_sig(v, 0.8 * (v**2 - (v**2).mean())), 0.8, rtol=0, atol=1e-12,\n              what=\"r = 0.8(x²−x̄²) ise imza 0.8\")\n        close(quad_sig(v, 0.8 * (v**2 - (v**2).mean()) + 2.0), 0.8, rtol=0, atol=1e-12,\n              what=\"sabit ekleme imzayı değiştirmemeli (merkezleme işi görür)\")\n        close(quad_sig(v, 3.0 * v), 0.0, rtol=0, atol=1e-12, what=\"doğrusal artıkta imza 0\")\n    R.test(\"quad_sig: merkezlenmiş x²'ye uydurur\", sig_ok)\n\n    def story():\n        t1, t2 = fit_poly(x, y, 1), fit_poly(x, y, 2)\n        r1 = phi(x, 1) @ t1 - y\n        s1, s2 = quad_sig(x, r1), quad_sig(x, resid(x, y, 2))\n        m1 = rmse(y, phi(x, 1) @ t1)\n        m2 = rmse(y, phi(x, 2) @ t2)\n        ensure(s1 < -0.5, f\"doğrusal uydurmanın U imzası {s1:.3f}: −0.8'e yakın negatif bekleniyordu\")\n        ensure(abs(s2) < 0.02, f\"derece 2'de imza {s2:.4f} ≈ 0 olmalıydı\")\n        ensure(abs(t2[2] - 0.8) < 0.15, f\"x² katsayısı {t2[2]:.3f}: 0.8'e yakın bekleniyordu\")\n        ensure(abs(t2[1] - 3) < 0.25 and abs(t2[0] + 1) < 0.4, f\"katsayılar {t2}: ≈ [−1, 3, 0.8] bekleniyordu\")\n        ensure(m2 < m1 / 3, f\"RMSE düşüşü yetersiz: {m1:.3f} → {m2:.3f} (~4 kat bekleniyordu)\")\n        r2lin = 1 - m1**2 / np.var(y)\n        ensure(r2lin > 0.9, f\"doğrusal R² {r2lin:.3f} — yanlış biçime rağmen ~0.9 kalmalıydı\")\n        return f\"imza {s1:.2f} → {s2:+.1e}; RMSE {m1:.3f} → {m2:.3f}; R²(1) ≈ {r2lin:.2f} ama biçim yanlıştı\"\n    R.test(\"U desenini kodla: imza ve RMSE hikâyesi\", story)\n"
}

In [ ]:
import importlib, labcheck, checks_m1_1_linreg as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Model: $\hat y = Xw + b$. Kayıp: MSE

$$
L(w,b)=\frac1n\sum_{i=1}^{n}\big(x_i^\top w+b-y_i\big)^2=\frac1n\lVert Xw+b-y\rVert^2 .
$$

Hata vektörü $e=Xw+b-y$ için zincir kuralı (M0.3) doğrudan verir:

$$
\nabla_w L=\frac2n X^\top e,\qquad \frac{\partial L}{\partial b}=\frac2n\sum_i e_i .
$$

İki yol var. **Kapalı form**: gradyanı sıfıra eşitleyip $(\tilde X^\top\tilde X)\theta=\tilde X^\top y$ normal denklemini çözmek ($\tilde X$, sonuna 1 sütunu eklenmiş $X$). **Gradient descent**: $\theta\leftarrow\theta-\eta\nabla L$ adımlarını tekrarlamak. Kapalı form tek seferde ama $d$ büyüyünce $O(d^3)$; GD ise sinir ağlarına genelleşen yol.

## 1 · Elle

<!-- çözüm -->

Üç noktayı bir koordinat düzlemine çiz: $(1,2)$, $(2,4)$, $(3,5)$. Başlangıç doğrusu $\hat y=0$, yani x ekseni.

Bir GD adımı at, sonra kapalı form çözümünü bul. Bir adım sonra $w=1.67$; en iyi değer $1.5$. İlk adım hedefi **aştı** — learning rate bu veri için büyük sayılır. (b) aşamasında bunu yüzlerce adım boyunca izleyeceksin.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Veri $x=[1,2,3]$, $y=[2,4,5]$; başlangıç $w=0$, $b=0$. Hatalar $e_i=\hat y_i-y_i$**

- `e1`: $e_1$
- `e2`: $e_2$
- `e3`: $e_3$
- `L`: $L=\frac1n\sum e_i^2$

**Gradyan ve bir adım, $\eta=0.1$**

- `gw`: $\partial L/\partial w=\frac2n\sum e_ix_i$
- `gb`: $\partial L/\partial b=\frac2n\sum e_i$
- `w1`: $w\leftarrow w-\eta\,\partial L/\partial w$
- `b1`: $b\leftarrow b-\eta\,\partial L/\partial b$

**Kapalı form: $w^*=\frac{\sum(x_i-\bar x)(y_i-\bar y)}{\sum(x_i-\bar x)^2}$, $b^*=\bar y-w^*\bar x$**

- `ws`: $w^*$
- `bs`: $b^*$
- `Ls`: En iyi kayıp $L(w^*,b^*)$

In [ ]:
cevaplar = dict(
    e1=None,
    e2=None,
    e3=None,
    L=None,
    gw=None,
    gb=None,
    w1=None,
    b1=None,
    ws=None,
    bs=None,
    Ls=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"e1":[-2,0],"e2":[-4,0],"e3":[-5,0],"L":[15,0],"gw":[-16.6667,0.01],"gb":[-7.3333,0.01],"w1":[1.66667,0.01],"b1":[0.73333,0.01],"ws":[1.5,0.001],"bs":[0.66667,0.005],"Ls":[0.055556,0.001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Tek özellikli ($x$ skaler) regresyonu listelerle yaz:

- `gd_step(xs, ys, w, b, lr)` → `(w_yeni, b_yeni, kayıp)`. Kayıp, güncellemeden **önceki** parametrelerle hesaplanır.
- `fit(xs, ys, lr=0.05, steps=2000)` → `(w, b)`; `gd_step`'i tekrar çağırır.
- `closed_form(xs, ys)` → `(w*, b*)`; yukarıdaki ortalama/kovaryans formülü.

Kontrol, `fit` sonucunun `closed_form` ile 4 basamak eşleşmesini bekler. `numpy` ve `torch` yasak.

In [ ]:
%%writefile stage_b.py
def gd_step(xs, ys, w, b, lr):
    """MSE için bir gradient descent adımı → (w_yeni, b_yeni, güncellemeden önceki kayıp)."""
    ...


def fit(xs, ys, lr=0.05, steps=2000):
    ...


def closed_form(xs, ys):
    ...


print(gd_step([1, 2, 3], [2, 4, 5], 0.0, 0.0, 0.1))   # (1.666…, 0.733…, 15.0)
print(fit([1, 2, 3], [2, 4, 5]))                      # ≈ (1.5, 0.667)
print(closed_form([1, 2, 3], [2, 4, 5]))              # (1.5, 0.666…)

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

Şimdi $d$ özellik. Örnekler üzerindeki döngü kalkıyor; **adım döngüsü kalıyor** (her adım bir öncekine bağlı, paralelleştirilemez). Kontrol, `normal_eq` içinde hiç döngü, `fit_gd` içinde tek katman döngü olmasını bekler.

- `normal_eq(X, y)` → `(w, b)`. `X`'e 1 sütunu ekle, `np.linalg.lstsq` (ya da `solve`) ile çöz. `np.linalg.inv` kullanma: sayısal olarak daha kötü.
- `fit_gd(X, y, lr=0.1, steps=500)` → `(w, b, losses)`; `losses` her adımdaki kayıp.
- `fit_df(df, target)` — pandas `DataFrame`'den `target` sütununu ayırıp `normal_eq` uygular; `w`'yi **özellik isimleriyle indekslenmiş** bir `pd.Series` olarak döndürür. İsimler, modelin hangi katsayının neye ait olduğunu unutmamasını sağlar.

In [ ]:
%%writefile stage_c.py
import numpy as np
import pandas as pd


def normal_eq(X, y):
    """→ (w: (d,), b: float). Döngü yok."""
    ...


def fit_gd(X, y, lr=0.1, steps=500):
    """→ (w, b, losses). Yalnız adım döngüsü."""
    ...


def fit_df(df, target):
    """→ (w: pd.Series [özellik isimleriyle], b)."""
    ...


rng = np.random.default_rng(0)
X = rng.normal(size=(100, 3))
y = X @ np.array([2.0, -1.0, 0.5]) + 3 + 0.1 * rng.normal(size=100)
print(normal_eq(X, y))
df = pd.DataFrame(X, columns=["alan", "yas", "kat"]).assign(fiyat=y)
print(fit_df(df, "fiyat"))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch'ta aynı model üç parça: `nn.Linear` (model), `nn.MSELoss` (kayıp), `torch.optim.SGD` (güncelleme kuralı). Eğitim döngüsü her zaman aynı beş satır:

```python
pred = model(X)              # ileri
loss = loss_fn(pred, y)      # kayıp
opt.zero_grad()              # eski gradyanları sil (yoksa birikir!)
loss.backward()              # geri
opt.step()                   # θ ← θ − η∇L
```

- `one_step_grads()` — elle örnekteki ($x=[1,2,3]$, $y=[2,4,5]$, $w=b=0$) $\partial L/\partial w$ ve $\partial L/\partial b$'yi autograd ile hesaplayıp `(gw, gb)` döndür. Elle bulduğun $-16.67$ ve $-7.33$ çıkmalı.
- `fit_torch(X, y, lr=0.1, steps=500)` — eğitilmiş `nn.Linear` döndür. `y` shape'ine dikkat: `(n,)` ile `(n, 1)` arasında broadcasting sessizce $(n, n)$ üretir ve kayıp yanlış olur.

In [ ]:
%%writefile stage_d.py
import torch
from torch import nn


def one_step_grads():
    """x=[1,2,3], y=[2,4,5], w=b=0 için (dL/dw, dL/db) — autograd ile."""
    x = torch.tensor([1., 2., 3.])
    y = torch.tensor([2., 4., 5.])
    w = torch.zeros((), requires_grad=True)
    b = torch.zeros((), requires_grad=True)
    ...


def fit_torch(X, y, lr=0.1, steps=500):
    """X: (n, d), y: (n,) → eğitilmiş nn.Linear(d, 1)."""
    model = nn.Linear(X.shape[1], 1)
    ...
    return model


print(one_step_grads())   # (tensor(-16.6667), tensor(-7.3333))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Hangi learning rate? Tipik cevap: birkaç değer dene. Döngüyle 64 lr × 200 adım = 12 800 küçük adım; her biri birkaç kernel çağrısı.

Bunun yerine 64 modeli **tek model** gibi eğit: ağırlıkları `W: (L, d)`, `b: (L,)` olarak tut. Tahmin `X @ W.T + b` → `(n, L)`: her sütun bir model. Kayıplar `(L,)`; `loss.sum().backward()` her modelin kendi gradyanını verir (sınama sorusuna bak). Güncellemede her model kendi lr'si ile ölçeklenir: `W -= lrs[:, None] * W.grad`.

- `sweep(X, y, lrs, steps=200)` → `(W, b, losses)`; `losses` son kayıplar `(L,)`.

Kontrol yalnız **adım döngüsüne** izin verir; lr üzerinde döngü yok. Sonuç, her lr ile tek tek eğitilmiş modellerle karşılaştırılır ve hız tablosu üretilir. Iraksayan (çok büyük lr) modellerin kaybı `inf`/`nan` olabilir; bu beklenen bir sonuç.

In [ ]:
%%writefile stage_e.py
import torch


def sweep(X, y, lrs, steps=200):
    """L = len(lrs) modeli birlikte eğit. → (W: (L, d), b: (L,), son kayıplar: (L,))
    Yalnız adım döngüsü; lr'ler üzerinde döngü yok."""
    L, d = lrs.shape[0], X.shape[1]
    W = torch.zeros(L, d, requires_grad=True)
    b = torch.zeros(L, requires_grad=True)
    ...


torch.manual_seed(0)
X = torch.randn(200, 4)
y = X @ torch.tensor([1.0, -2.0, 0.5, 0.0]) + 1.0
lrs = torch.logspace(-3, 0, 16)
W, b, losses = sweep(X, y, lrs)
print(losses)

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Polinom öznitelik genişletme: U desenini kodla

l4 deneyindeki tanıyı koda dök: artıkları U çizen veride eksik özniteliği genişlet. Veri $y = 3x + 0.8x^2 - 1 + \varepsilon$ ($n = 40$, $x \in [-2,2]$, $\sigma = 0.4$, tohum 0) — dosyanın dibinde hazır.

- `phi(x, deg)` → Vandermonde tasarım matrisi `(n, deg+1)`; sütunlar `1, x, …, x**deg`. Tek satır, döngü yok.
- `fit_poly(x, y, deg)` → `(deg+1,)` katsayılar; `np.linalg.lstsq` (inv yok).
- `rmse(y, y_hat)` → $\sqrt{\mathrm{mean}((\hat y - y)^2)}$.
- `resid(x, y, deg)` → uydurmanın artıkları `r = ŷ − y` (l4 deneyindeki işaret).
- `quad_sig(x, r)` → artıkları **merkezlenmiş** $x^2$'ye uyduran katsayı: $\frac{\sum z_i r_i}{\sum z_i^2}$, $z = x^2 - \overline{x^2}$. Merkezleme şart: $x^2$'nin ortalaması bias'a emilir, ham $x^2$ katsayıyı düşük gösterir (l4 notu).

Beklenen: derece 1'de imza ≈ −0.8 (eksik terim artıklarda kalır) ama R² ≈ 0.9'da kalır; derece 2'de imza ≈ 0, RMSE ~4 kat düşer ve katsayılar ≈ `[-1, 3, 0.8]` çıkar. Kontrol bu sayıları ve döngüsüzlüğü bekler. `numpy` serbest, `torch` yasak.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def phi(x, deg):
    """Vandermonde tasarım matrisi: (n,) → (n, deg+1); sütunlar 1, x, …, x**deg."""
    ...


def fit_poly(x, y, deg):
    """En küçük kareler: phi ile lstsq → (deg+1,) katsayılar."""
    ...


def rmse(y, y_hat):
    """√(mean((y_hat − y)²)) → float."""
    ...


def resid(x, y, deg):
    """Uydurmanın artıkları: r = ŷ − y (l4 deneyindeki işaret)."""
    ...


def quad_sig(x, r):
    """Artıkları MERKEZLENMİŞ x²'ye uyduran katsayı: U deseninin nicel imzası."""
    ...


rng = np.random.default_rng(0)
x = np.linspace(-2, 2, 40)
y = 3 * x + 0.8 * x**2 - 1 + 0.4 * rng.normal(size=40)

t1 = fit_poly(x, y, 1)
t2 = fit_poly(x, y, 2)
print(t1)                                        # [≈0.1, ≈3]: eğriyi ememiyor
print(t2)                                        # ≈ [-1.1, 3.0, 0.87]
print(rmse(y, phi(x, 1) @ t1), rmse(y, phi(x, 2) @ t2))   # 1.13 → 0.30
sig1 = quad_sig(x, resid(x, y, 1))
sig2 = quad_sig(x, resid(x, y, 2))
print(sig1, sig2)                                # ≈ −0.87, ≈ 0
R2 = 1 - rmse(y, phi(x, 1) @ t1) ** 2 / np.var(y)
print("R2 (doğrusal):", R2)                      # ≈ 0.91: yüksek ama biçim yanlış

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

## Makale ödevi: Golub & Reinsch (1970): SVD ve en küçük kareler

[Golub & Reinsch — Singular Value Decomposition and Least Squares Solutions (Numerische Mathematik 1970)](https://link.springer.com/article/10.1007/BF02163027)

En küçük kareleri dört yolla çöz: normal denklemler, QR (Householder), SVD ve kesilmiş SVD (küçük tekil değerleri at). Koşul sayısı 10²…10¹² olan tasarım matrislerinde (ör. Vandermonde / polinom regresyonu) çözüm hatasını ve kalıntıyı karşılaştır; normal denklemlerin hatanın κ², QR/SVD'nin κ ile büyüdüğünü göster. Kesilmiş SVD'nin ridge'e (M1.3) benzer düzenleyici etkisini tartış.


**Adımlar**

1. Kesin çözümü bilinen sentetik problemler kur (x* seç, b = Ax* + küçük gürültü).

**Kabul ölçütleri**

- Hata–κ grafiği: normal denklemler κ², QR/SVD κ.
- Kesilmiş SVD'nin gürültülü veride daha iyi genelleme örneği.

## Kaynaklar

- [Stanford CS229 — Lecture notes: Linear regression, LMS, normal equations](https://cs229.stanford.edu/main_notes.pdf) — doğrulanmadı
- [ISLP — An Introduction to Statistical Learning (Python), Bölüm 3](https://www.statlearning.com/) — doğrulanmadı
- [Dive into Deep Learning §3.1–3.5 Linear Regression (scratch + concise)](https://d2l.ai/chapter_linear-regression/index.html) — CC BY-SA 4.0 / MIT-0
- [PyTorch — nn.Linear, torch.optim.SGD](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) — BSD-3
- [Anscombe — Graphs in Statistical Analysis (The American Statistician 27(1), 1973)](https://doi.org/10.1080/00031305.1973.10478966) — doğrulanmadı